In [0]:
%sql
CREATE OR REPLACE TABLE segment_scores AS
WITH normalized AS (
    SELECT
        rs.segment_id,
        rs.corridor,
        rs.corridor_label,
        rs.midpoint_lon,
        rs.midpoint_lat,
        tc.aadt,
        tc.ev_daily_vehicles,
        tc.avg_speed_mph,
        rs.straightness_score,
        LEAST(100.0, (tc.ev_daily_vehicles / 1200.0) * 100.0) AS ev_demand_score,
        LEAST(100.0, (tc.aadt / 120000.0) * 100.0)            AS visibility_score,
        rs.straightness_score                                   AS geometry_score,
        (100.0 - wr.winter_risk_score)                         AS weather_score,
        GREATEST(0.0, 100.0 - (inc.annual_crashes_per_mile / 8.0) * 100.0) AS safety_score,
        CASE
            WHEN tc.avg_speed_mph BETWEEN 60 AND 75 THEN 100.0
            WHEN tc.avg_speed_mph < 60 THEN GREATEST(0.0, 100.0 - (60 - tc.avg_speed_mph) * 5.0)
            ELSE GREATEST(0.0, 100.0 - (tc.avg_speed_mph - 75) * 5.0)
        END AS speed_score
    FROM road_segments rs
    JOIN traffic_counts tc  ON tc.segment_id = rs.segment_id
    JOIN weather_risk   wr  ON wr.segment_id = rs.segment_id
    JOIN incidents      inc ON inc.segment_id = rs.segment_id
),
power_proximity AS (
    SELECT
        rs.segment_id,
        MIN(
            60.0 * SQRT(
                POWER((pi.lon - rs.midpoint_lon) * COS(RADIANS(rs.midpoint_lat)), 2)
                + POWER(pi.lat - rs.midpoint_lat, 2)
            )
        ) AS nearest_substation_miles
    FROM road_segments rs
    CROSS JOIN power_infra pi
    WHERE pi.type = 'substation'
    GROUP BY rs.segment_id
),
power_scored AS (
    SELECT segment_id,
        nearest_substation_miles,
        GREATEST(0.0, 100.0 - (nearest_substation_miles / 20.0) * 100.0) AS power_score
    FROM power_proximity
),
interchange_density AS (
    SELECT rs.segment_id,
        COUNT(ix.interchange_id) AS interchange_count
    FROM road_segments rs
    LEFT JOIN interchanges ix ON ix.segment_id = rs.segment_id
    GROUP BY rs.segment_id
),
interchange_scored AS (
    SELECT segment_id,
        interchange_count,
        GREATEST(0.0, 100.0 - interchange_count * 10.0) AS interchange_score
    FROM interchange_density
)
SELECT
    n.segment_id, n.corridor, n.corridor_label,
    n.midpoint_lon, n.midpoint_lat, n.aadt, n.ev_daily_vehicles,
    ps.nearest_substation_miles, ixs.interchange_count,
    ROUND(n.ev_demand_score,    1) AS ev_demand_score,
    ROUND(n.visibility_score,   1) AS visibility_score,
    ROUND(n.geometry_score,     1) AS geometry_score,
    ROUND(n.weather_score,      1) AS weather_score,
    ROUND(n.safety_score,       1) AS safety_score,
    ROUND(n.speed_score,        1) AS speed_score,
    ROUND(ps.power_score,       1) AS power_score,
    ROUND(ixs.interchange_score,1) AS interchange_score,
    ROUND(
        n.ev_demand_score    * 0.20
        + n.visibility_score * 0.08
        + n.geometry_score   * 0.10
        + n.weather_score    * 0.10
        + n.safety_score     * 0.15
        + n.speed_score      * 0.07
        + ps.power_score     * 0.12
        + ixs.interchange_score * 0.10
    , 2) AS composite_score
FROM normalized n
JOIN power_scored       ps  ON ps.segment_id = n.segment_id
JOIN interchange_scored ixs ON ixs.segment_id = n.segment_id
ORDER BY composite_score DESC

In [0]:
df = spark.table("segment_scores")
print(df.count())
df.show(3)

In [0]:
def haversine_miles(lon1, lat1, lon2, lat2):
    import math
    R = 3958.8
    lat1, lat2 = math.radians(lat1), math.radians(lat2)
    dlat = math.radians(lat2 - math.radians(math.degrees(lat2) - math.degrees(lat2)))
    dlon = math.radians(lon2 - lon1)
    dlat = lat2 - lat1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

# Convert to list of dicts for iteration
candidates = df.orderBy("composite_score", ascending=False).collect()

MIN_DISTANCE = 30
N_PILOTS = 4
selected = []
used_corridors = set()

for row in candidates:
    if len(selected) >= N_PILOTS:
        break
    
    if row["corridor"] in used_corridors:
        continue
    
    too_close = False
    for sel in selected:
        dist = haversine_miles(
            row["midpoint_lon"], row["midpoint_lat"],
            sel["midpoint_lon"], sel["midpoint_lat"]
        )
        if dist < MIN_DISTANCE:
            too_close = True
            break
    
    if not too_close:
        selected.append(row)
        used_corridors.add(row["corridor"])
        print(f"Selected: {row['segment_id']} [{row['corridor_label']}] score={row['composite_score']}")

print(f"\nTotal pilots selected: {len(selected)}")

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import monotonically_increasing_id

pilots = [Row(**{k: row[k] for k in row.asDict()}) for row in selected]
pilots_df = spark.createDataFrame(pilots)
pilots_df = pilots_df.withColumn("pilot_rank", monotonically_increasing_id() + 1)

pilots_df.write.mode("overwrite").saveAsTable("pilot_recommendations")

pilots_df.select("segment_id", "corridor_label", "composite_score", "pilot_rank").show()

In [0]:
# Convert to pandas and save as CSV
segments_pandas = df.toPandas()
pilots_pandas = pilots_df.toPandas()

segments_pandas.to_csv("/tmp/SEGMENTS.csv", index=False)
pilots_pandas.to_csv("/tmp/PILOTS.csv", index=False)

print("Files written to /tmp")
print(f"Segments: {len(segments_pandas)} rows")
print(f"Pilots: {len(pilots_pandas)} rows")

In [0]:
import shutil
import os

# Reload from Delta tables
df = spark.table("segment_scores")
pilots_df = spark.table("pilot_recommendations")

# Regenerate CSVs
segments_pandas = df.toPandas()
pilots_pandas = pilots_df.toPandas()

segments_pandas.to_csv("/tmp/SEGMENTS.csv", index=False)
pilots_pandas.to_csv("/tmp/PILOTS.csv", index=False)

# Copy to repo
os.makedirs("/Workspace/Users/jay.kline21@outlook.com/data-5035-2026/week07", exist_ok=True)

shutil.copy("/tmp/SEGMENTS.csv", "/Workspace/Users/jay.kline21@outlook.com/data-5035-2026/week07/SEGMENTS.csv")
shutil.copy("/tmp/PILOTS.csv", "/Workspace/Users/jay.kline21@outlook.com/data-5035-2026/week07/PILOTS.csv")

print("Done")

In [0]:
lines = [
    "# Week 07 — EV Charging Lane Pilot Location Selection",
    "",
    "## Overview",
    "This solution identifies the four best 10-mile highway segments for piloting",
    "an in-motion charging lane technology along two distinct Midwest highway corridors:",
    "I-70 between St. Louis and Kansas City, and the STL-Chicago corridor via",
    "I-55, I-57, I-64, and I-80.",
    "",
    "The selection balances demand, technical feasibility, safety, and strategic",
    "pilot value using two programming paradigms.",
    "",
    "## How to Run",
    "### Prerequisites",
    "- Databricks workspace with Unity Catalog",
    "- Python 3.x with PySpark",
    "",
    "### Step 1 — Generate Source Data",
    "Run notebook 01_generate_data. Creates seven Delta tables in your default schema:",
    "road_segments, traffic_counts, power_infra, interchanges, env_constraints, weather_risk, and incidents.",
    "",
    "### Step 2 — Score All Segments",
    "Run notebook 02_score_segments. Executes a declarative SQL pipeline using CTEs",
    "to score all 103 segments across nine dimensions and saves results as segment_scores.",
    "",
    "### Step 3 — Select Pilot Locations",
    "Run notebook 03_select_pilots. Applies a diversity-constrained selection algorithm",
    "to pick the four best pilots and saves them as pilot_recommendations.",
    "",
    "## Scoring Weights",
    "| Dimension | Weight | Notes |",
    "|-----------|--------|-------|",
    "| EV Demand | 20% | Daily EV vehicle count |",
    "| Crash Safety | 15% | Inverted crash rate per mile |",
    "| Power Infrastructure | 12% | Distance to nearest substation |",
    "| Road Geometry | 10% | Straightness score |",
    "| Weather Safety | 10% | Inverted winter risk score |",
    "| Interchange Density | 10% | Fewer interchanges = better |",
    "| Pilot Visibility | 8% | AADT as proxy for public exposure |",
    "| Environmental | 8% | Distance from wetlands/protected areas |",
    "| Speed Suitability | 7% | Optimal band 60-75 mph |",
    "",
    "## Programming Paradigms",
    "",
    "### Imperative (Python)",
    "Used for data generation and pilot selection. Both require explicit",
    "ordered steps and stateful logic that SQL cannot express naturally.",
    "Data generation builds each segment in order using position-dependent calculations.",
    "Pilot selection iterates through candidates and checks each one against",
    "previously selected sites for distance and corridor diversity.",
    "",
    "### Declarative (SQL)",
    "Used for scoring all 103 segments. SQL CTEs describe what each scoring",
    "dimension should look like without specifying how to compute it row by row.",
    "The query engine handles execution. Each CTE layer is independently testable.",
    "",
    "## Assumptions",
    "- Source data is synthetic, generated to match assignment schema and value ranges",
    "- Substation proximity uses equirectangular approximation for distance",
    "- Minimum 30-mile separation enforced between selected pilot sites",
    "- One pilot per corridor enforced to ensure geographic diversity",
]

path = "/Workspace/Users/jay.kline21@outlook.com/data-5035-2026/week07/README.md"
with open(path, "w") as f:
    f.write("\n".join(lines))

print("Done")

In [0]:
lines = [
    "# Reflection — Programming Paradigms",
    "",
    "## Which programming paradigm did you use where?",
    "I used two paradigms. Imperative Python handled data generation in notebook 01",
    "and pilot selection in notebook 03. Declarative SQL handled the scoring pipeline in notebook 02.",
    "",
    "## Why did you choose each paradigm in that area?",
    "Data generation had to be imperative because each segment's values depend on its location",
    "along the corridor. The urban factor drives range, EV share, crash rates, and straightness",
    "— all of which are calculated in sequence from position. There is no way to declare that",
    "logic as an output shape.",
    "",
    "Pilot selection is also imperative. Whether a segment gets selected depends on what was",
    "already selected before it. I iterated through candidates in score order, checked each one",
    "against all previously selected sites for distance and corridor, and added it only if it",
    "passes both checks. SQL can rank rows but cannot carry state from one row's decision into the next.",
    "",
    "SQL was the right fit for scoring because all 103 segments are processed identically.",
    "I used CTEs to break the scoring into logical layers, one per dimension, which makes each",
    "piece independently readable and testable. This structure also maps directly to how I would",
    "build a silver-layer transformation in our Databricks environment at work.",
    "",
    "## What additional data would improve your confidence?",
    "VMT-normalized crash rates would improve safety scoring. Crashes per lane-mile penalizes",
    "high-traffic segments that may actually be safer per trip. Monthly EV traffic counts would",
    "help because annual averages mask the winter demand peaks when anxiety about range is highest.",
    "",
    "## What political or operational risks exist?",
    "On I-70 in Missouri, the freight industry has a strong presence. A technology that reduces",
    "range issues for electric trucks could face opposition framed as enabling driver displacement,",
    "even if the pilot only targets passenger vehicles. Along stretches where local leaders are",
    "politically resistant to EV infrastructure, local officials could frame the pilot as forced",
    "adoption regardless of how it is positioned.",
    "",
    "Operationally, the 30 mph charging lane creates a 40 mph speed differential with adjacent",
    "traffic. That is one of the most dangerous conditions in highway design.",
]

path = "/Workspace/Users/jay.kline21@outlook.com/data-5035-2026/week07/REFLECTION.md"
with open(path, "w") as f:
    f.write("\n".join(lines))

print("Done")